# 🎵 Reto 10: Analizador de Plataforma de Streaming Musical

## Instituto Politécnico Nacional
### Programación para Ciencia de Datos

---

## 📋 Contexto del Reto

¡Felicidades! Has sido contratado como **Data Analyst** en **"SoundWave"**, una plataforma de streaming musical que compite con Spotify y Apple Music. Tu jefe te ha pedido analizar los datos de la plataforma para entender mejor el comportamiento de los usuarios y el rendimiento de los artistas.

```
    ╔══════════════════════════════════════════════════════════════╗
    ║                                                              ║
    ║   🎧 SOUNDWAVE ANALYTICS DASHBOARD 🎧                        ║
    ║                                                              ║
    ║   ┌─────────────┐  ┌─────────────┐  ┌─────────────┐         ║
    ║   │  USUARIOS   │  │  CANCIONES  │  │   STREAMS   │         ║
    ║   │   50,000+   │  │   100,000+  │  │  10M+ /día  │         ║
    ║   └─────────────┘  └─────────────┘  └─────────────┘         ║
    ║                                                              ║
    ║   Tu misión: Analizar y combinar datos para generar         ║
    ║   insights que impulsen el crecimiento de la plataforma     ║
    ║                                                              ║
    ╚══════════════════════════════════════════════════════════════╝
```

---

## 🎯 Objetivos de Aprendizaje

Al completar este reto, dominarás:

| Habilidad | Aplicación en el Reto |
|-----------|----------------------|
| `pd.concat()` | Combinar datos de múltiples meses |
| `pd.merge()` | Unir información de artistas, canciones y usuarios |
| `groupby()` | Calcular estadísticas por artista, género y país |
| `pivot_table()` | Crear reportes ejecutivos de streaming |
| `melt()` | Transformar datos para visualización |

---

## 📊 Datos Disponibles

Trabajarás con 5 conjuntos de datos que representan diferentes aspectos de la plataforma:

```
    ┌──────────────────────────────────────────────────────────────┐
    │                    MODELO DE DATOS                           │
    │                                                              │
    │   ┌─────────────┐         ┌─────────────┐                   │
    │   │  artistas   │────────▶│  canciones  │                   │
    │   │             │         │             │                   │
    │   │ artista_id  │         │ cancion_id  │                   │
    │   │ nombre      │         │ artista_id  │                   │
    │   │ pais_origen │         │ titulo      │                   │
    │   │ genero      │         │ duracion_s  │                   │
    │   └─────────────┘         │ fecha_pub   │                   │
    │                           └──────┬──────┘                   │
    │                                  │                          │
    │                                  ▼                          │
    │   ┌─────────────┐         ┌─────────────┐                   │
    │   │  usuarios   │────────▶│   streams   │                   │
    │   │             │         │             │                   │
    │   │ usuario_id  │         │ stream_id   │                   │
    │   │ nombre      │         │ usuario_id  │                   │
    │   │ pais        │         │ cancion_id  │                   │
    │   │ tipo_cuenta │         │ fecha       │                   │
    │   │ fecha_reg   │         │ completo    │                   │
    │   └─────────────┘         └─────────────┘                   │
    │                                                              │
    └──────────────────────────────────────────────────────────────┘
```

In [ ]:
# Importar librerías necesarias
import pandas as pd
import numpy as np

# Configuración para mejor visualización
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# DATOS DE LA PLATAFORMA SOUNDWAVE
# ═══════════════════════════════════════════════════════════════

# 1. CATÁLOGO DE ARTISTAS
artistas = pd.DataFrame({
    'artista_id': ['A001', 'A002', 'A003', 'A004', 'A005', 'A006', 'A007', 'A008'],
    'nombre': ['Bad Bunny', 'Taylor Swift', 'BTS', 'Peso Pluma', 'Dua Lipa', 
               'Feid', 'The Weeknd', 'Karol G'],
    'pais_origen': ['Puerto Rico', 'USA', 'Corea del Sur', 'México', 'Reino Unido',
                   'Colombia', 'Canadá', 'Colombia'],
    'genero_principal': ['Reggaeton', 'Pop', 'K-Pop', 'Regional Mexicano', 'Pop',
                        'Reggaeton', 'R&B', 'Reggaeton'],
    'seguidores_millones': [45.2, 52.1, 48.7, 12.3, 38.5, 18.9, 41.2, 35.6]
})

# 2. CATÁLOGO DE CANCIONES
canciones = pd.DataFrame({
    'cancion_id': ['C001', 'C002', 'C003', 'C004', 'C005', 'C006', 'C007', 'C008',
                   'C009', 'C010', 'C011', 'C012', 'C013', 'C014', 'C015', 'C016'],
    'artista_id': ['A001', 'A001', 'A002', 'A002', 'A003', 'A003', 'A004', 'A004',
                   'A005', 'A005', 'A006', 'A006', 'A007', 'A007', 'A008', 'A008'],
    'titulo': ['Monaco', 'Tití Me Preguntó', 'Anti-Hero', 'Shake It Off', 
               'Dynamite', 'Butter', 'Ella Baila Sola', 'Lady Gaga',
               'Levitating', 'Dance The Night', 'Ferxxo 100', 'Normal',
               'Blinding Lights', 'Save Your Tears', 'TQG', 'Provenza'],
    'duracion_segundos': [198, 243, 201, 219, 199, 165, 232, 185,
                          203, 176, 245, 198, 200, 215, 197, 208],
    'fecha_publicacion': ['2024-01-15', '2022-05-06', '2022-10-21', '2014-08-18',
                          '2020-08-21', '2021-05-21', '2023-03-10', '2023-06-23',
                          '2020-10-01', '2023-05-25', '2022-03-25', '2023-08-15',
                          '2020-03-20', '2021-02-05', '2023-02-24', '2022-04-22'],
    'explicito': [True, True, False, False, False, False, True, True,
                  False, False, True, True, False, False, True, False]
})

# 3. USUARIOS DE LA PLATAFORMA
usuarios = pd.DataFrame({
    'usuario_id': ['U001', 'U002', 'U003', 'U004', 'U005', 'U006', 'U007', 'U008',
                   'U009', 'U010', 'U011', 'U012', 'U013', 'U014', 'U015', 'U016'],
    'nombre': ['Carlos', 'María', 'Juan', 'Ana', 'Pedro', 'Sofía', 'Diego', 'Valentina',
               'Miguel', 'Camila', 'Andrés', 'Isabella', 'Ricardo', 'Fernanda', 'Jorge', 'Lucía'],
    'pais': ['México', 'México', 'Colombia', 'Colombia', 'España', 'España', 
             'Argentina', 'Argentina', 'Chile', 'Chile', 'Perú', 'Perú',
             'México', 'México', 'Colombia', 'Colombia'],
    'tipo_cuenta': ['Premium', 'Free', 'Premium', 'Premium', 'Free', 'Premium',
                    'Premium', 'Free', 'Premium', 'Free', 'Free', 'Premium',
                    'Premium', 'Free', 'Premium', 'Premium'],
    'edad': [22, 25, 19, 31, 28, 24, 35, 20, 27, 23, 29, 18, 33, 21, 26, 30],
    'fecha_registro': ['2023-01-15', '2023-03-22', '2022-11-08', '2023-06-17',
                       '2022-08-30', '2023-02-14', '2021-12-01', '2023-09-05',
                       '2022-05-20', '2023-07-11', '2023-04-03', '2022-10-25',
                       '2021-06-15', '2023-08-19', '2022-02-28', '2023-01-07']
})

# 4. STREAMS DE ENERO 2024
streams_enero = pd.DataFrame({
    'stream_id': [f'S{i:04d}' for i in range(1, 26)],
    'usuario_id': ['U001', 'U002', 'U001', 'U003', 'U004', 'U005', 'U006', 'U007',
                   'U008', 'U009', 'U010', 'U011', 'U012', 'U001', 'U002', 'U003',
                   'U013', 'U014', 'U015', 'U016', 'U004', 'U005', 'U006', 'U007', 'U008'],
    'cancion_id': ['C001', 'C003', 'C007', 'C005', 'C001', 'C009', 'C013', 'C015',
                   'C002', 'C011', 'C004', 'C006', 'C008', 'C010', 'C012', 'C014',
                   'C001', 'C003', 'C007', 'C015', 'C016', 'C002', 'C005', 'C009', 'C011'],
    'fecha': ['2024-01-05'] * 8 + ['2024-01-12'] * 8 + ['2024-01-20'] * 9,
    'escucha_completa': [True, True, False, True, True, True, True, False,
                         True, True, True, False, True, True, False, True,
                         True, True, True, True, False, True, True, True, False],
    'mes': ['Enero'] * 25
})

# 5. STREAMS DE FEBRERO 2024
streams_febrero = pd.DataFrame({
    'stream_id': [f'S{i:04d}' for i in range(26, 51)],
    'usuario_id': ['U002', 'U004', 'U006', 'U008', 'U010', 'U012', 'U014', 'U016',
                   'U001', 'U003', 'U005', 'U007', 'U009', 'U011', 'U013', 'U015',
                   'U002', 'U004', 'U006', 'U008', 'U010', 'U012', 'U014', 'U016', 'U001'],
    'cancion_id': ['C001', 'C001', 'C001', 'C003', 'C003', 'C005', 'C007', 'C007',
                   'C009', 'C011', 'C013', 'C015', 'C002', 'C004', 'C006', 'C008',
                   'C010', 'C012', 'C014', 'C016', 'C001', 'C007', 'C015', 'C003', 'C013'],
    'fecha': ['2024-02-03'] * 8 + ['2024-02-14'] * 8 + ['2024-02-25'] * 9,
    'escucha_completa': [True, True, True, False, True, True, True, True,
                         True, False, True, True, True, True, False, True,
                         True, True, True, False, True, True, True, True, True],
    'mes': ['Febrero'] * 25
})

print("✅ Datos cargados exitosamente")
print(f"\n📊 Resumen de datos disponibles:")
print(f"   • Artistas: {len(artistas)} registros")
print(f"   • Canciones: {len(canciones)} registros")
print(f"   • Usuarios: {len(usuarios)} registros")
print(f"   • Streams Enero: {len(streams_enero)} registros")
print(f"   • Streams Febrero: {len(streams_febrero)} registros")

---

## 🔥 Parte 1: Combinando Datos Históricos (concat)

### Ejercicio 1.1: Unificar Streams Mensuales

El equipo de analytics necesita un **dataset unificado** con todos los streams de Enero y Febrero para hacer análisis históricos.

**Tu tarea:**
1. Combinar `streams_enero` y `streams_febrero` en un solo DataFrame llamado `streams_total`
2. Reiniciar el índice para que sea consecutivo
3. Mostrar el número total de streams y las primeras 5 filas

```
    ANTES:                              DESPUÉS:
    
    streams_enero (25 filas)            streams_total (50 filas)
    ┌───────────────────────┐           ┌───────────────────────┐
    │ Enero streams...      │     +     │ Enero + Febrero       │
    └───────────────────────┘           │ combinados            │
    streams_febrero (25 filas)    ═══▶  │ índice reiniciado     │
    ┌───────────────────────┐           └───────────────────────┘
    │ Febrero streams...    │
    └───────────────────────┘
```

In [ ]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║ EJERCICIO 1.1: Combinar streams de Enero y Febrero            ║
# ╚═══════════════════════════════════════════════════════════════╝

# Apilar verticalmente los streams de ambos meses y reiniciar el índice
streams_total = pd.concat([streams_enero, streams_febrero], ignore_index=True)

# Verificación
print(f"Total de streams combinados: {len(streams_total)}")
print(streams_total.head())

---

## 🔗 Parte 2: Enriqueciendo Datos (merge)

### Ejercicio 2.1: Streams con Información de Canciones

Para el reporte semanal, necesitamos saber **qué canciones** se escucharon en cada stream.

**Tu tarea:**
1. Unir `streams_total` con `canciones` usando `cancion_id` como llave
2. El resultado debe llamarse `streams_canciones`
3. Usa un `inner join` (merge por defecto)

```
    streams_total              canciones
    ┌─────────────────┐       ┌─────────────────┐
    │ stream_id       │       │ cancion_id      │
    │ usuario_id      │       │ artista_id      │
    │ cancion_id  ────┼──────▶│ titulo          │
    │ fecha           │       │ duracion_seg    │
    │ escucha_completa│       │ fecha_pub       │
    └─────────────────┘       └─────────────────┘
```

In [ ]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║ EJERCICIO 2.1: Unir streams con información de canciones      ║
# ╚═══════════════════════════════════════════════════════════════╝

# Inner join entre los streams y el catálogo de canciones por cancion_id
streams_canciones = pd.merge(streams_total, canciones, on='cancion_id', how='inner')

# Verificación
print(streams_canciones[['stream_id', 'titulo', 'fecha', 'escucha_completa']].head(10))

### Ejercicio 2.2: Información Completa de Streams

Ahora el equipo de producto quiere un **dataset completo** que incluya información del artista.

**Tu tarea:**
1. Partiendo de `streams_canciones`, unir con `artistas` usando `artista_id`
2. Guardar resultado en `streams_completos`
3. Seleccionar solo las columnas: `stream_id`, `titulo`, `nombre` (del artista), `genero_principal`, `fecha`, `escucha_completa`

In [ ]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║ EJERCICIO 2.2: Agregar información de artistas                ║
# ╚═══════════════════════════════════════════════════════════════╝

# Unir el resultado anterior con artistas para sumar género y nombre del artista
streams_completos = pd.merge(streams_canciones, artistas, on='artista_id', how='inner')

# Columnas relevantes para el reporte (nombre = nombre del artista)
columnas_reporte = ['stream_id', 'titulo', 'nombre', 'genero_principal', 'fecha', 'escucha_completa']
streams_reporte = streams_completos[columnas_reporte]

# Verificación
print(streams_reporte.head(10))

### Ejercicio 2.3: Análisis de Usuarios Premium vs Free

El equipo comercial quiere comparar el comportamiento de usuarios **Premium vs Free**.

**Tu tarea:**
1. Unir `streams_total` con `usuarios` usando `usuario_id`
2. Calcular el porcentaje de escuchas completas por tipo de cuenta
3. ¿Qué tipo de usuario escucha más canciones completas?

```
    ┌─────────────────────────────────────────┐
    │     PREGUNTA DE NEGOCIO:                │
    │                                         │
    │     ¿Los usuarios Premium escuchan      │
    │     más canciones completas que los     │
    │     usuarios Free?                      │
    │                                         │
    │     💎 Premium: ___% completas          │
    │     🆓 Free: ___% completas             │
    └─────────────────────────────────────────┘
```

In [ ]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║ EJERCICIO 2.3: Comparar comportamiento Premium vs Free        ║
# ╚═══════════════════════════════════════════════════════════════╝

# 1. Unir streams con los datos de usuario (para conocer el tipo de cuenta)
streams_usuarios = pd.merge(streams_total, usuarios, on='usuario_id', how='inner')

# 2-3. Porcentaje de escuchas completas por tipo de cuenta
#    (el promedio de una columna booleana equivale a la proporción de True)
pct_completas = streams_usuarios.groupby('tipo_cuenta')['escucha_completa'].mean() * 100

print("Porcentaje de escuchas completas por tipo de cuenta:")
print(pct_completas.round(1))

# ¿Qué tipo de usuario escucha más completo?
tipo_top = pct_completas.idxmax()
print(f"\nLos usuarios {tipo_top} escuchan más canciones completas "
      f"({pct_completas.max():.1f}%).")

---

## 📈 Parte 3: Análisis Agregados (groupby)

### Ejercicio 3.1: Top Artistas por Streams

El equipo de contenido quiere saber cuáles son los **artistas más escuchados**.

**Tu tarea:**
1. Usando `streams_completos`, agrupa por nombre de artista
2. Cuenta el número total de streams por artista
3. Ordena de mayor a menor
4. Muestra el top 5

```
    🏆 TOP 5 ARTISTAS POR STREAMS
    ╔═══════════════════════════════════════╗
    ║  #1  │  ???????????  │  ?? streams   ║
    ║  #2  │  ???????????  │  ?? streams   ║
    ║  #3  │  ???????????  │  ?? streams   ║
    ║  #4  │  ???????????  │  ?? streams   ║
    ║  #5  │  ???????????  │  ?? streams   ║
    ╚═══════════════════════════════════════╝
```

In [ ]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║ EJERCICIO 3.1: Top 5 artistas más escuchados                  ║
# ╚═══════════════════════════════════════════════════════════════╝

# Contar streams por artista y quedarse con los 5 primeros
top_artistas = streams_completos.groupby('nombre').size() \
    .sort_values(ascending=False).head(5)

print("🏆 TOP 5 ARTISTAS POR STREAMS")
print("=" * 40)
for i, (artista, n) in enumerate(top_artistas.items(), 1):
    print(f"#{i}  {artista:<15} {n} streams")

### Ejercicio 3.2: Estadísticas por Género Musical

El equipo de playlists quiere entender qué géneros tienen mejor **engagement**.

**Tu tarea:**
1. Agrupa por `genero_principal`
2. Calcula:
   - Total de streams
   - Promedio de escuchas completas (tasa de completion)
   - Número de artistas únicos
3. Ordena por total de streams

💡 **Hint:** Usa `.agg()` con un diccionario para calcular múltiples estadísticas

In [ ]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║ EJERCICIO 3.2: Estadísticas por género musical                ║
# ╚═══════════════════════════════════════════════════════════════╝

# Múltiples agregaciones por género: total de streams, tasa de completion y artistas únicos
stats_genero = streams_completos.groupby('genero_principal').agg(
    total_streams=('stream_id', 'count'),
    tasa_completion=('escucha_completa', 'mean'),
    artistas_unicos=('nombre', 'nunique')
)
# Pasar la tasa a porcentaje y ordenar por total de streams
stats_genero['tasa_completion'] = (stats_genero['tasa_completion'] * 100).round(1)
stats_genero = stats_genero.sort_values('total_streams', ascending=False)

print("📊 Estadísticas por género musical:")
print(stats_genero)

### Ejercicio 3.3: Análisis por País y Mes

El equipo de expansión internacional quiere ver la **evolución del streaming por país**.

**Tu tarea:**
1. Primero, une `streams_total` con `usuarios` para obtener el país del usuario
2. Agrupa por `pais` y `mes`
3. Cuenta el número de streams
4. Muestra qué país creció más entre Enero y Febrero

In [ ]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║ EJERCICIO 3.3: Streams por país y mes                         ║
# ╚═══════════════════════════════════════════════════════════════╝

# 1-3. Unir con usuarios para tener el país y contar streams por país y mes
streams_pais = pd.merge(streams_total, usuarios, on='usuario_id', how='inner')
por_pais_mes = streams_pais.groupby(['pais', 'mes']).size().reset_index(name='streams')

print("🌎 Streams por país y mes:")
print(por_pais_mes)

# 4. País con mayor crecimiento entre Enero y Febrero
tabla = por_pais_mes.pivot(index='pais', columns='mes', values='streams').fillna(0)
tabla['crecimiento'] = tabla['Febrero'] - tabla['Enero']
pais_top = tabla['crecimiento'].idxmax()
print(f"\nEl país que más creció fue {pais_top} "
      f"(+{int(tabla.loc[pais_top, 'crecimiento'])} streams).")

---

## 📊 Parte 4: Reportes Ejecutivos (pivot_table)

### Ejercicio 4.1: Matriz de Streams por Género y País

El CEO quiere un **reporte visual** que muestre streams por género y país en formato de matriz.

**Tu tarea:**
1. Usa los datos combinados de streams con usuarios y canciones/artistas
2. Crea una pivot_table donde:
   - Filas: `genero_principal`
   - Columnas: `pais` (del usuario)
   - Valores: conteo de streams
3. Rellena los valores faltantes con 0

```
    📊 MATRIZ DE STREAMS
    
                    │ México │ Colombia │ España │ Argentina │ ...
    ────────────────┼────────┼──────────┼────────┼───────────┼────
    Reggaeton       │   ??   │    ??    │   ??   │    ??     │
    Pop             │   ??   │    ??    │   ??   │    ??     │
    K-Pop           │   ??   │    ??    │   ??   │    ??     │
    Regional Mex    │   ??   │    ??    │   ??   │    ??     │
    R&B             │   ??   │    ??    │   ??   │    ??     │
```

In [ ]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║ EJERCICIO 4.1: Pivot table de streams por género y país       ║
# ╚═══════════════════════════════════════════════════════════════╝

# 1-2. DataFrame con género (de artistas) y país (del usuario) en cada stream
streams_genero_pais = pd.merge(streams_completos, usuarios, on='usuario_id', how='inner')

# 3. Matriz género x país con conteo de streams; huecos rellenos con 0
pivot_genero_pais = pd.pivot_table(
    streams_genero_pais,
    values='stream_id',
    index='genero_principal',
    columns='pais',
    aggfunc='count',
    fill_value=0
)

print("📊 MATRIZ DE STREAMS (género x país):")
print(pivot_genero_pais)

### Ejercicio 4.2: Reporte de Engagement por Mes y Tipo de Cuenta

El equipo de producto quiere ver cómo varía el **engagement** (escuchas completas) entre usuarios Premium y Free a lo largo de los meses.

**Tu tarea:**
1. Crea una pivot_table donde:
   - Filas: `mes`
   - Columnas: `tipo_cuenta`
   - Valores: promedio de `escucha_completa`
2. Interpreta: ¿Mejoró el engagement en Febrero?

```
    📈 TASA DE COMPLETION POR MES Y TIPO DE CUENTA
    
              │  Premium  │   Free   │
    ──────────┼───────────┼──────────│
    Enero     │   ??%     │   ??%    │
    Febrero   │   ??%     │   ??%    │
```

In [ ]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║ EJERCICIO 4.2: Engagement por mes y tipo de cuenta            ║
# ╚═══════════════════════════════════════════════════════════════╝

# Reutilizamos streams_usuarios (streams + datos de usuario)
# Pivot: filas = mes, columnas = tipo de cuenta, valores = tasa de escucha completa
engagement = pd.pivot_table(
    streams_usuarios,
    values='escucha_completa',
    index='mes',
    columns='tipo_cuenta',
    aggfunc='mean'
)
# Expresar como porcentaje y ordenar los meses cronológicamente
engagement = (engagement * 100).round(1).reindex(['Enero', 'Febrero'])

print("📈 Tasa de completion por mes y tipo de cuenta (%):")
print(engagement)

# Interpretación: ¿mejoró el engagement en Febrero?
if (engagement.loc['Febrero'] >= engagement.loc['Enero']).all():
    print("\nEl engagement mejoró (o se mantuvo) en Febrero para ambos tipos de cuenta.")
else:
    print("\nEl engagement no mejoró de forma generalizada en Febrero.")

---

## 🔄 Parte 5: Transformación de Datos (melt) - BONUS

### Ejercicio 5.1: Preparar Datos para Visualización

El equipo de BI necesita los datos en **formato largo** para crear gráficos en su herramienta de visualización.

**Tu tarea:**
1. Toma la pivot_table del ejercicio 4.1 (género vs país)
2. Usa `melt()` para convertirla a formato largo
3. El resultado debe tener columnas: `genero_principal`, `pais`, `streams`

```
    FORMATO ANCHO (pivot):              FORMATO LARGO (melt):
    
            │ MX │ CO │ ES │            genero    │ pais │ streams
    ────────┼────┼────┼────│            ──────────┼──────┼────────
    Reggaeton│ 5  │ 3  │ 2 │    ═══▶   Reggaeton │  MX  │   5
    Pop     │ 4  │ 2  │ 3 │            Reggaeton │  CO  │   3
                                        Reggaeton │  ES  │   2
                                        Pop       │  MX  │   4
                                        ...       │ ...  │  ...
```

In [ ]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║ EJERCICIO 5.1 (BONUS): Convertir pivot a formato largo        ║
# ╚═══════════════════════════════════════════════════════════════╝

# 1. Reiniciar el índice para que 'genero_principal' sea una columna
pivot_reset = pivot_genero_pais.reset_index()

# 2. Pasar de formato ancho a largo con melt()
streams_largo = pd.melt(
    pivot_reset,
    id_vars=['genero_principal'],
    var_name='pais',
    value_name='streams'
)

print("🔄 Datos en formato largo (género, país, streams):")
print(streams_largo.head(15))

---

## 🎯 Desafío Final: Reporte Ejecutivo Completo

El CEO de SoundWave necesita un **reporte ejecutivo** para la junta de inversionistas. 

**Crea un análisis que responda estas preguntas:**

1. **Top 3 canciones más escuchadas** (con nombre de artista)
2. **Género con mejor tasa de completion**
3. **País con más streams totales**
4. **¿Los usuarios Premium escuchan más que los Free?** (streams por usuario)
5. **Artista con mejor crecimiento** entre Enero y Febrero

```
    ╔══════════════════════════════════════════════════════════════════╗
    ║                                                                  ║
    ║   📊 SOUNDWAVE - REPORTE EJECUTIVO Q1 2024                       ║
    ║                                                                  ║
    ║   ┌────────────────────────────────────────────────────────┐    ║
    ║   │ 🎵 TOP 3 CANCIONES                                     │    ║
    ║   │    1. ???????? - ????????                              │    ║
    ║   │    2. ???????? - ????????                              │    ║
    ║   │    3. ???????? - ????????                              │    ║
    ║   └────────────────────────────────────────────────────────┘    ║
    ║                                                                  ║
    ║   ┌────────────────────────────────────────────────────────┐    ║
    ║   │ 📈 MÉTRICAS CLAVE                                      │    ║
    ║   │    • Género líder: ???????? (??% completion)           │    ║
    ║   │    • País líder: ???????? (?? streams)                 │    ║
    ║   │    • Premium vs Free: ??? vs ??? streams/usuario       │    ║
    ║   └────────────────────────────────────────────────────────┘    ║
    ║                                                                  ║
    ║   ┌────────────────────────────────────────────────────────┐    ║
    ║   │ 🚀 ARTISTA DESTACADO                                   │    ║
    ║   │    ???????? - Mayor crecimiento mes a mes              │    ║
    ║   └────────────────────────────────────────────────────────┘    ║
    ║                                                                  ║
    ╚══════════════════════════════════════════════════════════════════╝
```

In [ ]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║ DESAFÍO FINAL: Genera el reporte ejecutivo completo           ║
# ╚═══════════════════════════════════════════════════════════════╝

print("="*60)
print("📊 SOUNDWAVE - REPORTE EJECUTIVO Q1 2024")
print("="*60)

# 1. Top 3 canciones más escuchadas (con nombre de artista)
print("\n🎵 TOP 3 CANCIONES")
print("-"*40)
top_canciones = streams_completos.groupby(['titulo', 'nombre']).size() \
    .sort_values(ascending=False).head(3)
for i, ((titulo, artista), n) in enumerate(top_canciones.items(), 1):
    print(f"{i}. {titulo} - {artista} ({n} streams)")

# 2. Género con mejor tasa de completion
print("\n🎸 GÉNERO CON MEJOR ENGAGEMENT")
print("-"*40)
completion_genero = streams_completos.groupby('genero_principal')['escucha_completa'].mean() * 100
genero_top = completion_genero.idxmax()
print(f"{genero_top} ({completion_genero.max():.1f}% completion)")

# 3. País con más streams totales
print("\n🌎 PAÍS LÍDER EN STREAMS")
print("-"*40)
streams_por_pais = streams_usuarios.groupby('pais').size().sort_values(ascending=False)
pais_lider = streams_por_pais.idxmax()
print(f"{pais_lider} ({streams_por_pais.max()} streams)")

# 4. Premium vs Free (streams por usuario)
print("\n💎 PREMIUM VS FREE (streams por usuario)")
print("-"*40)
# Streams totales por tipo de cuenta entre número de usuarios de ese tipo
streams_por_tipo = streams_usuarios.groupby('tipo_cuenta').size()
usuarios_por_tipo = usuarios.groupby('tipo_cuenta').size()
promedio_por_usuario = (streams_por_tipo / usuarios_por_tipo).round(2)
for tipo, valor in promedio_por_usuario.items():
    print(f"{tipo}: {valor} streams/usuario")

# 5. Artista con mayor crecimiento entre Enero y Febrero
print("\n🚀 ARTISTA DESTACADO (mayor crecimiento)")
print("-"*40)
por_artista_mes = streams_completos.groupby(['nombre', 'mes']).size() \
    .unstack(fill_value=0)
por_artista_mes['crecimiento'] = por_artista_mes.get('Febrero', 0) - por_artista_mes.get('Enero', 0)
artista_destacado = por_artista_mes['crecimiento'].idxmax()
print(f"{artista_destacado} (+{int(por_artista_mes.loc[artista_destacado, 'crecimiento'])} "
      f"streams de Enero a Febrero)")

print("\n" + "="*60)

---

## ✅ Criterios de Evaluación

| Criterio | Puntos | Descripción |
|----------|--------|-------------|
| **Parte 1: concat** | 15 | Combinar correctamente los DataFrames de streams |
| **Parte 2: merge** | 25 | Realizar joins correctos entre tablas |
| **Parte 3: groupby** | 25 | Cálculos agregados correctos |
| **Parte 4: pivot_table** | 20 | Crear reportes en formato matriz |
| **Parte 5: melt** | 5 | (BONUS) Transformar datos correctamente |
| **Desafío Final** | 10 | Análisis completo y correcto |
| **Total** | **100** | |

---

## 🎓 Recursos Adicionales

Si te trabas en algún ejercicio, consulta:

- [Documentación de pd.concat()](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)
- [Documentación de pd.merge()](https://pandas.pydata.org/docs/reference/api/pandas.merge.html)
- [Documentación de groupby()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)
- [Documentación de pivot_table()](https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html)
- [Documentación de melt()](https://pandas.pydata.org/docs/reference/api/pandas.melt.html)

---

**¡Éxito en tu análisis! 🎵📊**